**Main 2 Stage Pipeline for analyse Citation relevance**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

True

In [3]:
from prediction.predict_main import *
from utils.citation_context import *
from utils.generate_report import *
import traceback as tr
from tqdm import tqdm

/Work/conda-envs/fire-mixsub/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MAX_REFERENCES = 32 # Prediction is in GPU, to quickly test take only few references
TEST_SET_MAIN_PAPERS = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/main-paper-set-1.txt"
FACET_PREDICTION_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions"

**Load Main papers from Paper Index**

In [5]:
paper_index = load_paper_index(max_files=4096) # turn off the max-files if you want to load all papers from dataset
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 4095line [00:01, 3392.01line/s]


Loaded 4096 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (201.8 MB in memory)
Total 4096 Main papers


**Load Main Paper (M) IDs - This is Train Set**

We avoid these IDs for Test

In [6]:
with open(TEST_SET_MAIN_PAPERS) as f:
    test_set_main_paper_ids = [line.strip() for line in f if line.strip()]
Nm = len(test_set_main_paper_ids)
print(f"Loaded {Nm} main paper ids [To be excluded]")

Loaded 59 main paper ids [To be excluded]


### Prediction Block for Single M ###

In [7]:
citation_ctx_extractor = CitationContextExtractor()

**Main Model Loading [FACET]**

In [8]:
narrative_local_dir = os.environ.get(
        "NARRATIVE_MODEL_LOCAL_DIR", str(project_root / "model" / "narrative"))
context_classifier_local_dir = os.environ.get(
    "CONTEXT_CLASSIFIER_MODEL_LOCAL_DIR", str(project_root / "model" / "context_classifier"))

facet_predictor = CitationPipelinePredictor(
    narrative_local_dir=narrative_local_dir,
    context_classifier_local_dir=context_classifier_local_dir,
    narrative_checkpoint_kind="train-best",
    narrative_wandb_version="latest",
    narrative_version="v2",
    context_checkpoint_kind="train-best",
    context_wandb_version="latest",
)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/parthosarothi/.netrc.


Looking up 'qwen-narrative-adapter-train-best:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: narrative-model/051026-123828/train_best
  Downloading: narrative-model/051026-123828/train_best/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/README.md
  Downloading: narrative-model/051026-123828/train_best/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/adapter_config.json
  Downloading: narrative-model/051026-123828/train_best/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/adapter_model.safetensors
  Downloading: narrative-model/051026-123828/train_best/adapter_model/added_tokens.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/a

Loading weights: 100%|██████████| 339/339 [01:33<00:00,  3.64it/s]


✓ Loaded prompt from 'qwen-context-classifier-adapter-train-best:latest:v50' metadata (version=v1, output_keys=['FUNCTION', 'M_ASPECT', 'POLARITY', 'EVIDENCE', 'R_MATCH'])
Looking up 'qwen-context-classifier-adapter-train-best:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: context-classifier/051026-123828/train_best
  Downloading: context-classifier/051026-123828/train_best/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adapter_model/README.md
  Downloading: context-classifier/051026-123828/train_best/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adapter_model/adapter_config.json
  Downloading: context-classifier/051026-123828/train_best/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adap

In [9]:
import random

**Pick a single random test paper**

In [15]:
while True:
    rand_idx = random.randint(0,nb_main_papers-1)
    m_id = all_main_paper_ids[rand_idx]
    if m_id not in test_set_main_paper_ids:
        break
print(f"Random test main paper seletected {m_id}")

Random test main paper seletected 2312.11879


**Pick a set of random test papers**

In [20]:
T = 25
test_set = []
while len(test_set) < T:
    rand_idx = random.randint(0,nb_main_papers-1)
    m_id = all_main_paper_ids[rand_idx]
    if m_id not in test_set_main_paper_ids:
        test_set.append(m_id)
        print(f"Random test main paper seletected {m_id}")
with open(f"/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/M_test_set.txt","w") as f:
    for m_id in test_set:
        f.write(f"{m_id}\n")
f.close()

Random test main paper seletected 2303.11329
Random test main paper seletected hep-th/0404091
Random test main paper seletected 2305.16709
Random test main paper seletected 2302.13788
Random test main paper seletected 2303.17156
Random test main paper seletected 2308.11796
Random test main paper seletected 2305.09028
Random test main paper seletected 2311.15895
Random test main paper seletected 2302.02931
Random test main paper seletected 2302.11636
Random test main paper seletected 2305.16058
Random test main paper seletected 2302.02607
Random test main paper seletected 2306.16803
Random test main paper seletected 2307.05891
Random test main paper seletected 2311.06493
Random test main paper seletected hep-th/0404230
Random test main paper seletected 2311.01007
Random test main paper seletected 2304.14887
Random test main paper seletected 2310.06347
Random test main paper seletected 2309.14779
Random test main paper seletected 2312.00486
Random test main paper seletected 2304.08288
Ra

**Get Main Paper content from S3**

In [16]:
m_data = paper_index[m_id] # main paper data from paper index file
m_title = m_data["title"]
print(f'Main Title: {m_title}')
main_paper_data = get_paper_from_s3(m_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
_,m_abstract = get_title_and_abstract(main_paper_data)
print("Retrieved abstract...Extracting Aspects/Narrative from abstract")

Main Title: The rotating solutions beyond the spontaneous scalarization in
  Einstein-Maxwell-scalar theory
Retrieved main paper data from S3...Extracting titles (not needed) and abstract
Retrieved abstract...Extracting Aspects/Narrative from abstract


In [12]:
m_abstract

'The conversion of protons to positrons at the horizon of a black hole (BH) is\nconsidered. It is shown that the process may efficiently proceed for BHs with\nmasses in the range $\\sim 10^{18}$ -- $10^{21}$ g. It is argued that the\nelectric charge of BH acquired by the proton accretion to BH could create\nelectric field near BH horizon close to the critical Schwinger one. It leads to\nefficient electron-positron pair production, when electrons are back captured\nby the BH while positrons are emitted into outer space. Annihilation of these\npositrons with electrons in the interstellar medium may at least partially\nexplain the origin of the observed 511 keV line.'

**Additional Prediction for Main paper**

In [17]:
m_narrative = facet_predictor.predict_narrative(m_abstract,max_new_tokens=2048)
print("Gathering all citation context")
status,cite_ctx = find_citation_context(m_id,bulk_paper_index=paper_index,citation_context_extractor=citation_ctx_extractor)
ref_contexts = cite_ctx[m_id]["references"] # all citation contexts

Gathering all citation context
Extracted context for 2312.11879 success [32/32]


**Predict on Reference Paper Data [Batch]**

In [ ]:
import time

t0 = time.time()
in_corpus_refs_ids = m_data["in_corpus_references"] # this comes from paper index data
in_corpus_refs_ids = in_corpus_refs_ids[:min(len(in_corpus_refs_ids),MAX_REFERENCES)]
ref_papers_data = await get_papers_from_s3(in_corpus_refs_ids)

references = await facet_predictor.predict_references(
    m_title=m_title, m_narrative=m_narrative,
    ref_contexts=ref_contexts, ref_papers_data=ref_papers_data,
    batch_size=8, narrative_max_new_tokens=2048, relation_max_new_tokens=512,
)
facet_relevance_batched = {"m_id":m_id,"m_title":m_title,"m_abstract":m_abstract,"m_aspects":m_narrative,"references":references}

elapsed = time.time() - t0
print(f"Batched pipeline: {len(references)} references -- wall time {elapsed:.1f}s")

with open(f"{FACET_PREDICTION_DIR}/{m_id}.json","w") as f:
    json.dump(facet_relevance_batched,f,indent=3)

# generate report
out_path = f"{FACET_PREDICTION_DIR}/reports/{m_id}_report.md"
with open(out_path, "w") as f:
    f.write(generate_report(facet_relevance_batched))
print(f"Wrote report -> {out_path}")

Fetching papers: 100%|██████████| 32/32 [00:04<00:00,  7.61it/s]


32/32 references usable for batching


Classifying citation relations: 100%|██████████| 48/48 [01:03<00:00,  1.32s/context]

Batched pipeline: 32 references -- wall time 218.4s
Wrote report -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/2312.11879_report.md


### Embedding Computation ##

**Unload Predictor Model**

In [5]:
import gc
import torch

print(f"GPU memory before unload: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# facet_predictor.narrative_model.model IS facet_predictor.context_model.model
# (one shared base + both LoRA adapters -- see CitationPipelinePredictor), so
# dropping the single top-level reference is enough; nothing else in this
# notebook holds a separate reference to the model/tokenizer.
del facet_predictor
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after unload : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

GPU memory before unload: 0.00 GB allocated, 0.00 GB reserved


NameError: name 'facet_predictor' is not defined

In [7]:
from prediction.qwen_embedding import *

In [9]:
embedding_qwen = AspectSimilarity(model_name="Qwen/Qwen3-Embedding-4B")

Loading weights: 100%|██████████| 398/398 [00:07<00:00, 52.73it/s] 


In [13]:
FACET_PREDICTION_FILE = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/2312.11879.json"
embedding_qwen.run(input_file=FACET_PREDICTION_FILE)


aspect similarity: 100%|██████████| 1/1 [00:05<00:00,  5.41s/it]
